# Introduction
We will use this notebook to explore the data initially, before initiating any trading algorithm.

# Import

In [1]:
# Standard library imports
import datetime as dt
import os
import sys

# Third party imports
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Local imports
from utilities import simple_returns, log_returns, directional_positions, cross_sectional_positions, position_returns, date_returns, calculate_sr
from ar_script import rolling_ar_forecast
from ar_script import multiple_forecasts
from ar_script import multiple_scores
from ar_script import rolling_regression

# Data Analysis

In [2]:
df_train = pd.read_csv('df_train.csv')

In [3]:
df_train.head()

,date,symbol,open,close,low,high,volume
0,2010-01-04,ACTS,15.13,14.97,14.84,15.28,5459.882
1,2010-01-04,AMWD,7.00,7.12,6.98,7.14,92067.275
2,2010-01-04,ARV,18.27,18.31,18.12,18.47,38034.257
3,2010-01-04,BBY,3.11,3.11,3.09,3.13,40964.820
4,2010-01-04,BCDM,19.78,19.53,19.41,19.90,3646.991


In [4]:
df_train.tail()

,date,symbol,open,close,low,high,volume
100595,2013-12-31,XTG,20.15,20.00,20.15,20.15,42044.703
100596,2013-12-31,YPN,14.94,14.82,14.94,14.94,29030.106
100597,2013-12-31,YRD,3.40,3.39,3.40,3.40,11778.336
100598,2013-12-31,YVNL,4.25,4.24,4.25,4.25,6431.843
100599,2013-12-31,ZQN,32.01,32.04,32.01,32.01,54319.469


In [5]:
df_train.shape

(100600, 7)

In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100600 entries, 0 to 100599
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   date    100600 non-null  object 
 1   symbol  100600 non-null  object 
 2   open    100600 non-null  float64
 3   close   100600 non-null  float64
 4   low     100600 non-null  float64
 5   high    100600 non-null  float64
 6   volume  100600 non-null  float64
dtypes: float64(5), object(2)
memory usage: 5.4+ MB


In [7]:
# Unique symbols
print('Number of unique symbols:', len(df_train['symbol'].unique()))
print('Number of unique dates:', len(df_train['date'].unique()))

Number of unique symbols: 100
Number of unique dates: 1006


In [8]:
df_train_wide = df_train.pivot(index='date', values='close', columns='symbol')
df_train_wide.index = pd.to_datetime(df_train_wide.index)
df_train_wide = df_train_wide.sort_index()

In [9]:
print("Total null values:", df_train_wide.isna().sum().sum())

Total null values: 0


In [10]:
# Summary statistics
df_train_wide.describe()

symbol,ACTS,AMWD,ARV,BBY,BCDM,BZK,BZQM,CDM,CDRX,CLYQ,...,WYLR,WZB,XBRQ,XFG,XPT,XTG,YPN,YRD,YVNL,ZQN
count,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,...,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.00000
mean,14.077087,8.004970,18.001382,2.724473,20.434463,15.859861,45.251769,10.887932,15.543757,2.452187,...,12.670805,6.507097,6.060924,18.747684,70.564215,18.543280,12.897783,4.293101,2.475835,25.14830
std,1.847735,1.227604,2.517147,0.320843,3.058197,2.410707,4.933455,1.370540,3.697120,0.230995,...,1.677963,0.770862,1.947529,2.655176,12.242660,1.894617,1.757839,0.727431,0.512197,2.58555
min,10.280000,5.380000,12.830000,2.160000,14.720000,11.350000,32.740000,8.100000,9.950000,1.860000,...,8.470000,4.800000,2.830000,13.030000,44.860000,14.190000,9.380000,2.210000,1.310000,19.57000
25%,12.700000,7.040000,16.457500,2.480000,18.077500,14.052500,42.177500,9.760000,12.450000,2.270000,...,11.560000,5.960000,4.180000,16.862500,60.425000,17.262500,11.532500,3.920000,2.120000,23.48250
50%,14.160000,7.895000,17.770000,2.660000,20.095000,15.560000,45.160000,10.590000,14.270000,2.450000,...,12.540000,6.340000,6.195000,18.590000,70.065000,18.735000,12.780000,4.330000,2.540000,25.15500
75%,14.970000,8.820000,18.887500,2.950000,22.290000,17.430000,48.165000,12.250000,18.600000,2.620000,...,13.890000,6.970000,7.487500,20.210000,77.745000,19.667500,13.917500,4.820000,2.760000,26.72750
max,19.910000,11.170000,29.870000,3.600000,29.770000,22.570000,57.940000,14.070000,23.570000,3.110000,...,16.880000,9.150000,11.250000,26.750000,102.610000,23.950000,17.350000,5.860000,4.240000,32.23000


In [11]:
# Split into train-test sets
df_train_start = df_train_wide[df_train_wide.index < "2013-01-01"]

In [12]:
K_STEP_RETURNS = 1

In [13]:
# Calculate log returns
df_train_log_returns = log_returns(df_train_start, K_STEP_RETURNS, False)
# Calculate simple returns
df_train_simple_returns = simple_returns(df_train_start, K_STEP_RETURNS, False)

In [14]:
df_train_simple_returns.head(5)

symbol,ACTS,AMWD,ARV,BBY,BCDM,BZK,BZQM,CDM,CDRX,CLYQ,...,WYLR,WZB,XBRQ,XFG,XPT,XTG,YPN,YRD,YVNL,ZQN
date,,,,,,,,,,,,,,,,,,,,,
2010-01-05,-0.008016,-0.018258,-0.002731,-0.006431,-0.011777,-0.002642,-0.022172,-0.008410,-0.014243,-0.014035,...,-0.008922,-0.014658,-0.003099,0.001000,-0.009442,-0.012768,-0.012857,-0.001912,-0.006623,-0.002379
2010-01-06,0.000673,-0.021459,-0.020811,-0.016181,-0.015544,-0.012583,-0.017054,0.006168,-0.015209,-0.003559,...,-0.012003,-0.016529,-0.014508,-0.031968,-0.020049,0.004139,-0.007959,-0.015326,-0.030000,-0.022480
2010-01-07,-0.010767,-0.024854,0.016219,-0.013158,-0.011579,-0.003353,-0.003352,-0.000766,-0.026255,-0.014286,...,0.010630,0.001681,-0.011567,-0.022704,0.040751,0.003091,0.003647,-0.005837,-0.010309,-0.005575
2010-01-08,-0.020408,-0.010495,-0.023665,-0.023333,-0.004792,0.000000,-0.014441,-0.010736,-0.026963,0.010870,...,-0.013524,-0.011745,-0.018085,-0.029567,-0.020303,-0.006163,-0.013808,-0.005871,-0.041667,-0.028732
2010-01-11,0.010417,0.012121,0.008455,-0.013652,0.004815,-0.017497,0.021678,0.013953,-0.007335,0.007168,...,0.001523,-0.005093,-0.022752,-0.014690,-0.004441,0.007752,0.011791,-0.015748,-0.025362,-0.010823


In [15]:
# Identify symbols
symbols = sorted(df_train['symbol'].unique().tolist())

In [16]:
# Calculate AR(1) forecasts for simple returns - no intercept
ar1_simple_rets_forecasts = multiple_forecasts(df_train_simple_returns, 
                                               symbols, AutoReg, 252, 
                                               K_STEP_RETURNS, lags=[1], trend='n')
# Calculate R2 scores
ar1_simple_rets_r2 = multiple_scores(ar1_simple_rets_forecasts, 
                                     symbols, r2_score, 'r2')

In [17]:
# Concatenate to have a long dataframe
new_names_forecasts = []
for i, curr_symbol in enumerate(symbols):
    curr_forecast = ar1_simple_rets_forecasts[i]
    curr_renamed_forecast = curr_forecast.copy().rename(columns={curr_symbol: 'return'})
    curr_renamed_forecast['symbol'] = curr_symbol 
    new_names_forecasts.append(curr_renamed_forecast)
all_forecasts = pd.concat(new_names_forecasts, axis=0).sort_index()


### Strategy 1: Directional AR(1) positions
In this strategy we use the forecasted value of each instrument to instantiate directional positions. Specifically, we go long asset i at time t if the predicted value is positive, otherwise we go short, with arbitrary tie breaking since we don't care when forecasted values are 0. 

In [18]:
ar1_forecast_direct = directional_positions(all_forecasts, 'forecast')
ar1_forecast_pos_return = position_returns(ar1_forecast_direct, 'weight', 'return')
ar1_forecast_date_rets = date_returns(ar1_forecast_pos_return, 'positions_return')
ar1_forecast_sr = calculate_sr(ar1_forecast_date_rets, 'positions_return')
print("Daily SR on directional positions: ", ar1_forecast_sr)

Daily SR on directional positions:  0.026017658557665222


### Strategt 2: Cross-sectional AR(1) positions
In this strategy we will use the forecasted values of each instrument returns to instantiate cross-sectional positions. Specifically, we will rank each instrument by the forecasted value, and we will go long the top k instruments with the largest forecasted values, and short the bottom k instruments with the lowest forecasted values. Initially, $k=5$.

In [24]:
ar1_forecast_cross = cross_sectional_positions(all_forecasts, 'forecast', 10)
ar1_forecast_cross_ret = position_returns(ar1_forecast_cross, 'weight', 'return')
ar1_forecast_cross_date_rets = date_returns(ar1_forecast_cross_ret, 'positions_return')
ar1_forecast_cross_sr = calculate_sr(ar1_forecast_cross_date_rets, 'positions_return')
print("Daily SR on cross-sectional positions", ar1_forecast_cross_sr)

Daily SR on cross-sectional positions 0.21413654032944388
